# 🌊 GNN pour la Modélisation des Écoulements de Fluides Incompressibles
## Projet M1 — Polymathic AI / The WELL Dataset

---

**Objectifs :** Implémenter et comparer trois architectures GNN (GNS/MeshGraphNet, E(3)-GNN, GMN) pour l'émulation d'écoulements de fluides incompressibles à partir des données *The WELL* de Polymathic AI.

**Structure du notebook :**
1. ⚙️ Configuration et Installation
2. 📦 Chargement et Exploration des Données
3. 🔧 Pré-traitement et Construction des Graphes
4. 🧠 Architecture GNS / MeshGraphNet
5. 🔄 Architecture E(3)-GNN (Équivariant)
6. ⚛️ Architecture GMN (Physics-Informed)
7. 🏋️ Entraînement et Validation
8. 📊 Évaluation et Visualisation

---
**Références :**
- Sanchez-Gonzalez et al. (2020) — *Learning to Simulate Complex Physics with Graph Networks* (GNS)
- Pfaff et al. (2021) — *Learning Mesh-Based Simulation with Graph Networks* (MeshGraphNet)
- Satorras et al. (2021) — *E(n) Equivariant Graph Neural Networks* (EGNN)
- Ohana et al. (2024) — *The Well: a Large-Scale Collection of Diverse Physics Simulations*

---
## Section 1 — ⚙️ Configuration et Installation

In [ ]:
# Installation des dépendances (exécuter une seule fois)
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# !pip install torch-geometric
# !pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
# !pip install h5py scipy matplotlib seaborn tqdm wandb e3nn
# !pip install the-well  # Package officiel Polymathic AI

# Vérification de l'environnement
import sys
print(f"Python {sys.version}")

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice utilisé : {DEVICE}")

In [ ]:
# Imports principaux
import os
import json
import time
import random
import numpy as np
import h5py
import scipy
from pathlib import Path
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

import torch_geometric
from torch_geometric.data import Data, DataLoader, Dataset
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from tqdm.auto import tqdm

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Style des plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print("✅ Imports réussis")

In [ ]:
# Configuration globale du projet
@dataclass
class Config:
    # Données
    DATA_DIR: str = './data'
    DATASET: str = 'shear_flow'
    NX: int = 128       # Résolution spatiale x
    NY: int = 128       # Résolution spatiale y
    DT: float = 0.01    # Pas de temps physique
    T_TRAIN: int = 50   # Longueur des trajectoires d'entraînement
    T_EVAL: int = 100   # Longueur d'évaluation (rollout)
    
    # Graphe
    CONNECTIVITY: str = '4-connected'  # '4-connected', '8-connected', 'radius'
    RADIUS: float = 0.05               # Rayon pour connectivity='radius'
    
    # Modèle
    HIDDEN_DIM: int = 128
    NUM_LAYERS: int = 10
    NODE_IN: int = 3    # (vx, vy, p)
    EDGE_IN: int = 3    # (dx, dy, dist)
    NODE_OUT: int = 2   # (dvx, dvy) accélérations
    
    # Entraînement
    BATCH_SIZE: int = 4
    LR: float = 1e-4
    EPOCHS: int = 100
    NOISE_STD: float = 3e-4
    NOISE_STEPS: int = 3
    GRAD_CLIP: float = 1.0
    
    # Logging
    LOG_EVERY: int = 10
    SAVE_DIR: str = './experiments/checkpoints'

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
os.makedirs('./results/figures', exist_ok=True)
print("✅ Configuration chargée")
print(json.dumps(cfg.__dict__, indent=2))

---
## Section 2 — 📦 Chargement et Exploration des Données

**The WELL** est accessible via le package `the-well` ou directement depuis HuggingFace / AWS S3.
Dans ce notebook, nous montrons les deux approches.

In [ ]:
# ============================================================
# Approche 1 : Interface officielle The WELL
# ============================================================
try:
    from the_well.data import WELLDataModule
    
    # Initialisation du module de données
    data_module = WELLDataModule(
        dataset_name='shear_flow',
        data_dir=cfg.DATA_DIR,
        batch_size=cfg.BATCH_SIZE,
        num_workers=4,
        download=True,  # Télécharge automatiquement si non présent
    )
    data_module.setup()
    
    # Accès aux loaders
    train_loader_raw = data_module.train_dataloader()
    val_loader_raw   = data_module.val_dataloader()
    test_loader_raw  = data_module.test_dataloader()
    
    print("✅ The WELL chargé via l'interface officielle")
    USING_WELL_API = True

except ImportError:
    print("⚠️ Package 'the-well' non disponible — utilisation des données HDF5 locales")
    USING_WELL_API = False

In [ ]:
# ============================================================
# Approche 2 : Données HDF5 locales (ou générées synthétiquement)
# ============================================================

def generate_synthetic_shear_flow(
    n_trajectories: int = 50,
    T: int = 100,
    Nx: int = 64,
    Ny: int = 64,
    save_path: Optional[str] = None
) -> Dict[str, np.ndarray]:
    """
    Génère des données synthétiques d'écoulement de cisaillement pour les tests.
    
    Note: En situation réelle, remplacer par les vraies données The WELL.
    Cette fonction simule un écoulement de cisaillement simplifié.
    """
    print(f"Génération de {n_trajectories} trajectoires synthétiques ({Nx}x{Ny}, T={T})...")
    
    x = np.linspace(0, 2*np.pi, Nx, endpoint=False)
    y = np.linspace(0, 2*np.pi, Ny, endpoint=False)
    X, Y = np.meshgrid(x, y, indexing='ij')
    
    all_vx = np.zeros((n_trajectories, T, Nx, Ny), dtype=np.float32)
    all_vy = np.zeros((n_trajectories, T, Nx, Ny), dtype=np.float32)
    all_p  = np.zeros((n_trajectories, T, Nx, Ny), dtype=np.float32)
    
    for traj in range(n_trajectories):
        # Conditions initiales aléatoires
        eps = np.random.uniform(0.01, 0.1)
        Re  = np.random.uniform(500, 2000)
        nu  = 1.0 / Re
        
        # Profil de vitesse de base (cisaillement)
        vx_base = np.tanh((Y - np.pi) / 0.5)
        
        # Perturbation initiale
        perturbation = eps * np.random.randn(Nx, Ny)
        
        vx = vx_base + perturbation
        vy = eps * np.sin(X + np.random.randn(Nx, Ny) * 0.1)
        p  = -0.5 * vx**2 + eps * np.random.randn(Nx, Ny) * 0.01
        
        # Évolution temporelle simplifiée (diffusion + advection linéarisée)
        for t in range(T):
            all_vx[traj, t] = vx
            all_vy[traj, t] = vy
            all_p[traj, t]  = p
            
            # Schéma d'Euler simplifié (pour la démonstration)
            dt = 0.01
            dvx = -nu * (np.roll(vx,-1,0) - 2*vx + np.roll(vx,1,0)) / (2*np.pi/Nx)**2
            dvy = -nu * (np.roll(vy,-1,1) - 2*vy + np.roll(vy,1,1)) / (2*np.pi/Ny)**2
            
            vx = vx + dt * (dvx + np.random.randn(Nx, Ny) * 0.001)
            vy = vy + dt * (dvy + np.random.randn(Nx, Ny) * 0.001)
            p  = -0.5 * (vx**2 + vy**2)
    
    data = {
        'velocity_x': all_vx,
        'velocity_y': all_vy,
        'pressure': all_p,
        'metadata': {
            'Nx': Nx, 'Ny': Ny, 'T': T,
            'n_trajectories': n_trajectories
        }
    }
    
    if save_path:
        with h5py.File(save_path, 'w') as f:
            for key, val in data.items():
                if isinstance(val, dict):
                    grp = f.create_group(key)
                    for k, v in val.items():
                        grp.attrs[k] = v
                else:
                    f.create_dataset(key, data=val, compression='gzip')
        print(f"✅ Données sauvegardées dans {save_path}")
    
    return data


# Génération ou chargement des données
os.makedirs(cfg.DATA_DIR, exist_ok=True)
train_file = f"{cfg.DATA_DIR}/shear_flow_train.hdf5"
val_file   = f"{cfg.DATA_DIR}/shear_flow_val.hdf5"

if not os.path.exists(train_file):
    train_data = generate_synthetic_shear_flow(
        n_trajectories=80, T=cfg.T_TRAIN, Nx=cfg.NX, Ny=cfg.NY,
        save_path=train_file
    )
else:
    with h5py.File(train_file, 'r') as f:
        train_data = {
            'velocity_x': f['velocity_x'][:],
            'velocity_y': f['velocity_y'][:],
            'pressure':   f['pressure'][:],
        }
    print(f"✅ Données train chargées depuis {train_file}")

if not os.path.exists(val_file):
    val_data = generate_synthetic_shear_flow(
        n_trajectories=20, T=cfg.T_EVAL, Nx=cfg.NX, Ny=cfg.NY,
        save_path=val_file
    )
else:
    with h5py.File(val_file, 'r') as f:
        val_data = {
            'velocity_x': f['velocity_x'][:],
            'velocity_y': f['velocity_y'][:],
            'pressure':   f['pressure'][:],
        }
    print(f"✅ Données val chargées depuis {val_file}")

In [ ]:
# ============================================================
# Exploration et Visualisation des Données
# ============================================================

vx = train_data['velocity_x']
vy = train_data['velocity_y']
p  = train_data['pressure']

print("=" * 50)
print("STATISTIQUES DU DATASET")
print("=" * 50)
print(f"Trajectoires d'entraînement : {vx.shape[0]}")
print(f"Pas de temps par trajectoire : {vx.shape[1]}")
print(f"Résolution spatiale : {vx.shape[2]} × {vx.shape[3]}")
print(f"\nStatistiques vx : μ={vx.mean():.3f}, σ={vx.std():.3f}, min={vx.min():.3f}, max={vx.max():.3f}")
print(f"Statistiques vy : μ={vy.mean():.3f}, σ={vy.std():.3f}, min={vy.min():.3f}, max={vy.max():.3f}")
print(f"Statistiques p  : μ={p.mean():.3f}, σ={p.std():.3f}, min={p.min():.3f}, max={p.max():.3f}")

# Visualisation d'une trajectoire
traj_idx = 0
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Évolution temporelle de l\'écoulement de cisaillement', fontsize=14, fontweight='bold')

time_steps = [0, 10, 30, 49]
fields = [
    (vx[traj_idx], 'Vitesse $u_x$', 'RdBu_r'),
    (vy[traj_idx], 'Vitesse $u_y$', 'RdBu_r'),
    (p[traj_idx],  'Pression $p$',  'viridis'),
]

for row, (field, label, cmap) in enumerate(fields):
    for col, t in enumerate(time_steps):
        ax = axes[row, col]
        im = ax.imshow(field[t].T, cmap=cmap, origin='lower', aspect='equal')
        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_title(f't = {t}', fontsize=10)
        if col == 0:
            ax.set_ylabel(label, fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('./results/figures/data_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Calcul et visualisation du spectre d'énergie cinétique
def kinetic_energy_spectrum(u: np.ndarray, v: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Spectre de l'énergie cinétique en fonction du nombre d'onde k."""
    Nx, Ny = u.shape
    u_hat = np.fft.fft2(u) / (Nx * Ny)
    v_hat = np.fft.fft2(v) / (Nx * Ny)
    E_hat = 0.5 * (np.abs(u_hat)**2 + np.abs(v_hat)**2)
    
    kx = np.fft.fftfreq(Nx) * Nx
    ky = np.fft.fftfreq(Ny) * Ny
    K = np.sqrt(kx[:,None]**2 + ky[None,:]**2)
    
    k_max = int(min(Nx, Ny) / 2)
    E_k = np.zeros(k_max)
    counts = np.zeros(k_max)
    
    for k in range(1, k_max):
        mask = (K >= k - 0.5) & (K < k + 0.5)
        E_k[k] = E_hat[mask].sum()
        counts[k] = mask.sum()
    
    return np.arange(k_max), E_k


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Spectre au cours du temps
ax = axes[0]
t_indices = [0, 10, 25, 49]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(t_indices)))

for t, color in zip(t_indices, colors):
    k_vals, E_k = kinetic_energy_spectrum(vx[0, t], vy[0, t])
    ax.loglog(k_vals[2:], E_k[2:], color=color, label=f't={t}', alpha=0.8)

# Loi de Kolmogorov k^{-5/3}
k_ref = np.array([2, 20])
ax.loglog(k_ref, 0.01 * k_ref**(-5/3), 'k--', label=r'$k^{-5/3}$ (Kolmogorov)', lw=2)
ax.legend(fontsize=9)
ax.set_xlabel('Nombre d\'onde k')
ax.set_ylabel('E(k)')
ax.set_title('Spectre d\'énergie cinétique')

# Énergie cinétique totale au cours du temps
ax = axes[1]
for traj in range(min(5, vx.shape[0])):
    E_t = 0.5 * (vx[traj]**2 + vy[traj]**2).mean(axis=(1,2))
    ax.plot(E_t, alpha=0.7)
ax.set_xlabel('Pas de temps t')
ax.set_ylabel('Énergie cinétique moyenne')
ax.set_title('Évolution de l\'énergie cinétique')

# Vorticité ω = ∂vy/∂x - ∂vx/∂y
ax = axes[2]
vort = (np.roll(vy[0,49], -1, axis=0) - np.roll(vy[0,49], 1, axis=0)) / 2 - \
       (np.roll(vx[0,49], -1, axis=1) - np.roll(vx[0,49], 1, axis=1)) / 2
im = ax.imshow(vort.T, cmap='RdBu_r', origin='lower')
plt.colorbar(im, ax=ax)
ax.set_title('Champ de vorticité (t=49)')
ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('./results/figures/spectral_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — 🔧 Pré-traitement et Construction des Graphes

In [ ]:
# ============================================================
# Normalisation des données
# ============================================================

class Normalizer:
    """Normalise et dénormalise les données de simulation."""
    
    def __init__(self):
        self.mean = None
        self.std  = None
    
    def fit(self, data: np.ndarray):
        """Calcule la moyenne et l'écart-type."""
        self.mean = data.mean()
        self.std  = data.std() + 1e-8
        return self
    
    def transform(self, data):
        return (data - self.mean) / self.std
    
    def inverse_transform(self, data):
        return data * self.std + self.mean
    
    def fit_transform(self, data):
        return self.fit(data).transform(data)

# Calcul des statistiques sur l'ensemble d'entraînement
norm_vx = Normalizer().fit(train_data['velocity_x'])
norm_vy = Normalizer().fit(train_data['velocity_y'])
norm_p  = Normalizer().fit(train_data['pressure'])

print("Statistiques de normalisation :")
print(f"vx : μ={norm_vx.mean:.4f}, σ={norm_vx.std:.4f}")
print(f"vy : μ={norm_vy.mean:.4f}, σ={norm_vy.std:.4f}")
print(f"p  : μ={norm_p.mean:.4f}, σ={norm_p.std:.4f}")

In [ ]:
# ============================================================
# Construction du graphe
# ============================================================

def build_grid_edges(Nx: int, Ny: int, connectivity: str = '4-connected') -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Construit les arêtes d'une grille 2D avec conditions périodiques.
    
    Args:
        Nx, Ny      : Résolution de la grille
        connectivity: '4-connected', '8-connected'
    
    Returns:
        edge_index  : (2, num_edges) indices des arêtes
        edge_attr   : (num_edges, 3) attributs (dx, dy, distance)
    """
    if connectivity == '4-connected':
        offsets = [(1,0), (-1,0), (0,1), (0,-1)]
    elif connectivity == '8-connected':
        offsets = [(1,0), (-1,0), (0,1), (0,-1),
                   (1,1), (1,-1), (-1,1), (-1,-1)]
    else:
        raise ValueError(f"Connectivity inconnue : {connectivity}")
    
    src_list, dst_list, attr_list = [], [], []
    
    for i in range(Nx):
        for j in range(Ny):
            src = i * Ny + j
            for di, dj in offsets:
                ni = (i + di) % Nx  # Periodicité
                nj = (j + dj) % Ny
                dst = ni * Ny + nj
                src_list.append(src)
                dst_list.append(dst)
                dx = di / Nx
                dy = dj / Ny
                dist = np.sqrt(dx**2 + dy**2)
                attr_list.append([dx, dy, dist])
    
    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    edge_attr  = torch.tensor(attr_list, dtype=torch.float)
    return edge_index, edge_attr


# Pré-calculer les arêtes (structure fixe du graphe)
print("Construction des arêtes du graphe...")
EDGE_INDEX, EDGE_ATTR = build_grid_edges(cfg.NX, cfg.NY, cfg.CONNECTIVITY)
print(f"✅ Graphe construit :")
print(f"   Nœuds : {cfg.NX * cfg.NY}")
print(f"   Arêtes : {EDGE_INDEX.shape[1]}")
print(f"   Degré moyen : {EDGE_INDEX.shape[1] / (cfg.NX * cfg.NY):.1f}")

In [ ]:
# ============================================================
# Dataset PyTorch Geometric
# ============================================================

class ShearFlowDataset(Dataset):
    """
    Dataset PyTorch Geometric pour l'écoulement de cisaillement.
    
    Chaque échantillon est un pair (état_t, état_t+1) d'une trajectoire,
    représenté comme un graphe.
    """
    
    def __init__(
        self,
        data: Dict[str, np.ndarray],
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        norm_vx: Normalizer,
        norm_vy: Normalizer,
        norm_p: Normalizer,
        noise_std: float = 0.0,
        noise_steps: int = 0,
        dt: float = 0.01,
    ):
        super().__init__()
        self.vx = data['velocity_x']
        self.vy = data['velocity_y']
        self.p  = data['pressure']
        self.edge_index = edge_index
        self.edge_attr  = edge_attr
        self.norm_vx = norm_vx
        self.norm_vy = norm_vy
        self.norm_p  = norm_p
        self.noise_std   = noise_std
        self.noise_steps = noise_steps
        self.dt = dt
        
        self.N_traj = self.vx.shape[0]
        self.T      = self.vx.shape[1]
        self.Nx     = self.vx.shape[2]
        self.Ny     = self.vx.shape[3]
        
        # Index de tous les paires (traj, t) valides
        self.indices = [
            (traj, t)
            for traj in range(self.N_traj)
            for t in range(self.T - 1)
        ]
    
    def len(self):
        return len(self.indices)
    
    def get(self, idx: int) -> Data:
        traj, t = self.indices[idx]
        
        # Récupération des champs
        vx_t  = self.norm_vx.transform(self.vx[traj, t]).flatten()
        vy_t  = self.norm_vy.transform(self.vy[traj, t]).flatten()
        p_t   = self.norm_p.transform(self.p[traj, t]).flatten()
        vx_t1 = self.norm_vx.transform(self.vx[traj, t+1]).flatten()
        vy_t1 = self.norm_vy.transform(self.vy[traj, t+1]).flatten()
        
        # Features des nœuds (état courant)
        x = np.stack([vx_t, vy_t, p_t], axis=1).astype(np.float32)
        
        # Bruit de marche aléatoire (robustesse au rollout)
        if self.noise_std > 0 and self.noise_steps > 0:
            noise = np.random.randn(*x.shape) * self.noise_std
            for _ in range(self.noise_steps - 1):
                noise += np.random.randn(*x.shape) * self.noise_std
            x = x + noise
        
        # Cible : accélération (dérivée temporelle)
        target_vx = (vx_t1 - vx_t) / self.dt
        target_vy = (vy_t1 - vy_t) / self.dt
        y = np.stack([target_vx, target_vy], axis=1).astype(np.float32)
        
        return Data(
            x=torch.from_numpy(x),
            y=torch.from_numpy(y),
            edge_index=self.edge_index,
            edge_attr=self.edge_attr,
            traj=traj,
            t=t,
        )


# Création des datasets
train_dataset = ShearFlowDataset(
    train_data, EDGE_INDEX, EDGE_ATTR,
    norm_vx, norm_vy, norm_p,
    noise_std=cfg.NOISE_STD,
    noise_steps=cfg.NOISE_STEPS,
    dt=cfg.DT,
)
val_dataset = ShearFlowDataset(
    val_data, EDGE_INDEX, EDGE_ATTR,
    norm_vx, norm_vy, norm_p,
    noise_std=0.0,  # Pas de bruit en validation
    dt=cfg.DT,
)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ Datasets créés :")
print(f"   Train : {len(train_dataset)} échantillons")
print(f"   Val   : {len(val_dataset)} échantillons")

# Test d'un échantillon
sample = train_dataset[0]
print(f"\nÉchantillon exemple :")
print(f"   x (features nœuds) : {sample.x.shape}")
print(f"   y (cible accél.)   : {sample.y.shape}")
print(f"   edge_index         : {sample.edge_index.shape}")
print(f"   edge_attr          : {sample.edge_attr.shape}")

---
## Section 4 — 🧠 Architecture GNS / MeshGraphNet

In [ ]:
# ============================================================
# MLP helper
# ============================================================

def build_mlp(
    in_dim: int,
    out_dim: int,
    hidden_dim: int = 128,
    n_hidden: int = 2,
    activation = nn.SiLU,
    layer_norm: bool = True,
) -> nn.Sequential:
    """Construit un MLP avec couches cachées et normalisation optionnelle."""
    layers = []
    current_dim = in_dim
    
    for _ in range(n_hidden):
        layers.extend([nn.Linear(current_dim, hidden_dim), activation()])
        if layer_norm:
            layers.append(nn.LayerNorm(hidden_dim))
        current_dim = hidden_dim
    
    layers.append(nn.Linear(current_dim, out_dim))
    return nn.Sequential(*layers)


# ============================================================
# Couche de Message Passing (InteractionNetwork)
# ============================================================

class InteractionLayer(MessagePassing):
    """
    Couche de passage de messages style GNS/MeshGraphNet.
    Met à jour les représentations des nœuds ET des arêtes.
    """
    
    def __init__(self, node_dim: int, edge_dim: int, hidden_dim: int):
        super().__init__(aggr='sum')
        
        # φ_e : fonction de message (arête)
        self.edge_mlp = build_mlp(
            in_dim=2*node_dim + edge_dim,
            out_dim=edge_dim,
            hidden_dim=hidden_dim,
            layer_norm=True
        )
        # φ_v : fonction de mise à jour (nœud)
        self.node_mlp = build_mlp(
            in_dim=node_dim + edge_dim,
            out_dim=node_dim,
            hidden_dim=hidden_dim,
            layer_norm=True
        )
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Calcul des messages sur les arêtes
        i, j = edge_index
        edge_input = torch.cat([x[i], x[j], edge_attr], dim=-1)
        new_edge_attr = edge_attr + self.edge_mlp(edge_input)  # Résidu
        
        # Agrégation et mise à jour des nœuds
        aggr = self.propagate(edge_index, x=x, edge_attr=new_edge_attr)
        new_x = x + self.node_mlp(torch.cat([x, aggr], dim=-1))  # Résidu
        
        return new_x, new_edge_attr
    
    def message(self, x_j: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        return edge_attr  # Messages = attributs d'arêtes mis à jour


# ============================================================
# MeshGraphNet complet
# ============================================================

class MeshGraphNet(nn.Module):
    """
    MeshGraphNet (Pfaff et al., 2021) : simulateur de physique sur maillage.
    Architecture Encode → Process (L couches) → Decode.
    """
    
    def __init__(
        self,
        node_in: int,
        edge_in: int,
        node_out: int,
        hidden_dim: int = 128,
        num_layers: int = 10,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Encodeurs
        self.node_encoder = build_mlp(node_in, hidden_dim, hidden_dim)
        self.edge_encoder = build_mlp(edge_in, hidden_dim, hidden_dim)
        
        # Processeur : L couches de message passing
        self.layers = nn.ModuleList([
            InteractionLayer(hidden_dim, hidden_dim, hidden_dim)
            for _ in range(num_layers)
        ])
        
        # Décodeur
        self.decoder = build_mlp(hidden_dim, node_out, hidden_dim, layer_norm=False)
    
    def forward(self, data: Data) -> torch.Tensor:
        x = self.node_encoder(data.x)       # (N, hidden)
        e = self.edge_encoder(data.edge_attr)  # (E, hidden)
        
        for layer in self.layers:
            x, e = layer(x, data.edge_index, e)
        
        return self.decoder(x)  # (N, node_out)
    
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Test de l'architecture
model_mgn = MeshGraphNet(
    node_in=cfg.NODE_IN,
    edge_in=cfg.EDGE_IN,
    node_out=cfg.NODE_OUT,
    hidden_dim=cfg.HIDDEN_DIM,
    num_layers=cfg.NUM_LAYERS,
).to(DEVICE)

print("=" * 50)
print("ARCHITECTURE MESHGRAPHNET")
print("=" * 50)
print(model_mgn)
print(f"\nParamètres entraînables : {model_mgn.count_parameters():,}")

# Test forward pass
sample_batch = next(iter(train_loader)).to(DEVICE)
with torch.no_grad():
    out = model_mgn(sample_batch)
print(f"\nTest forward pass : x={sample_batch.x.shape} → y={out.shape} ✅")

---
## Section 5 — 🔄 Architecture E(3)-GNN (Équivariant)

In [ ]:
# ============================================================
# EGNN : Equivariant Graph Neural Network
# Satorras et al. (2021)
# ============================================================

class EGNNLayer(nn.Module):
    """
    Couche EGNN équivariante sous E(n).
    
    Maintient deux types de représentations :
    - h_i : features invariantes (scalaires)
    - r_i : coordonnées équivariantes (vecteurs)
    """
    
    def __init__(self, d: int, hidden: int = 64, coords_agg: str = 'mean'):
        super().__init__()
        self.coords_agg = coords_agg
        
        # φ_e : fonction de message invariante
        self.phi_e = nn.Sequential(
            nn.Linear(2*d + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden),  nn.SiLU(),
            nn.LayerNorm(hidden)
        )
        # φ_h : mise à jour des features invariantes
        self.phi_h = nn.Sequential(
            nn.Linear(d + hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, d)
        )
        # φ_x : poids pour mise à jour équivariante des coordonnées
        self.phi_x = nn.Sequential(
            nn.Linear(hidden, 1)
        )
        # Gate pour atténuer les mises à jour de coordonnées
        self.gate = nn.Sequential(
            nn.Linear(hidden, 1), nn.Sigmoid()
        )
    
    def forward(
        self,
        h: torch.Tensor,
        x: torch.Tensor,
        edge_index: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        src, dst = edge_index
        N = h.shape[0]
        
        # Vecteurs de déplacement et distances au carré
        r_ij = x[src] - x[dst]                              # (E, 2)
        d_sq = (r_ij**2).sum(dim=-1, keepdim=True)          # (E, 1)
        
        # Messages invariants
        m_ij = self.phi_e(torch.cat([h[src], h[dst], d_sq], dim=-1))  # (E, hidden)
        
        # Agrégation des messages vers les nœuds destination
        agg = torch.zeros(N, m_ij.shape[-1], device=h.device)
        agg.scatter_add_(0, dst.unsqueeze(-1).expand_as(m_ij), m_ij)
        
        # Mise à jour des features (invariante)
        h_new = h + self.phi_h(torch.cat([h, agg], dim=-1))
        
        # Mise à jour équivariante des coordonnées
        weight = self.phi_x(m_ij) * self.gate(m_ij)         # (E, 1)
        coord_msg = r_ij * weight                            # (E, 2) — équivariant!
        coord_agg = torch.zeros(N, x.shape[-1], device=x.device)
        coord_agg.scatter_add_(0, dst.unsqueeze(-1).expand_as(coord_msg), coord_msg)
        
        if self.coords_agg == 'mean':
            # Normaliser par le degré
            degree = torch.zeros(N, 1, device=x.device)
            degree.scatter_add_(0, dst.unsqueeze(-1), torch.ones(len(src), 1, device=x.device))
            degree = degree.clamp(min=1)
            coord_agg = coord_agg / degree
        
        x_new = x + coord_agg
        return h_new, x_new


class EGNN(nn.Module):
    """
    Réseau EGNN complet pour la simulation de fluides.
    Préserve l'équivariance sous translations et rotations E(n).
    """
    
    def __init__(
        self,
        node_in: int,
        node_out: int,
        hidden_dim: int = 64,
        num_layers: int = 6,
        coord_dim: int = 2,
    ):
        super().__init__()
        self.coord_dim = coord_dim
        
        # Encodeur de features invariantes (scalaires uniquement)
        scalar_in = node_in - coord_dim  # Enlever les composantes vectorielles (vx, vy)
        self.feature_encoder = build_mlp(scalar_in + 1, hidden_dim, hidden_dim)
        # +1 pour la magnitude de vitesse (invariante)
        
        # Couches EGNN
        self.layers = nn.ModuleList([
            EGNNLayer(hidden_dim, hidden_dim)
            for _ in range(num_layers)
        ])
        
        # Décodeur (prédit des scalaires, puis projette sur vecteurs via coordonnées)
        self.decoder_scalar = build_mlp(hidden_dim, 1, hidden_dim, layer_norm=False)
        self.decoder_norm   = build_mlp(hidden_dim, 1, hidden_dim, layer_norm=False)
    
    def forward(self, data: Data) -> torch.Tensor:
        # Séparation features scalaires et vectorielles
        vx = data.x[:, 0:1]   # Composante x vitesse
        vy = data.x[:, 1:2]   # Composante y vitesse
        p  = data.x[:, 2:3]   # Pression (scalaire)
        
        # Coordonnées équivariantes : position dans le champ de vitesse
        x = torch.cat([vx, vy], dim=-1)  # (N, 2) — équivariant
        
        # Features invariantes : pression + magnitude de vitesse
        speed = (vx**2 + vy**2).sqrt()  # Invariant sous rotation
        h_in  = torch.cat([p, speed], dim=-1)  # (N, 2)
        
        # Encodage
        h = self.feature_encoder(h_in)  # (N, hidden)
        
        # Couches équivariantes
        for layer in self.layers:
            h, x = layer(h, x, data.edge_index)
        
        # Décodage : prédire les accélérations
        # Accélération = scalaire * direction_équivariante
        accel_norm = self.decoder_norm(h)   # Magnitude (scalaire, invariant)
        x_norm = x / (x.norm(dim=-1, keepdim=True) + 1e-8)  # Direction normalisée
        accel = accel_norm * x_norm         # Vecteur équivariant
        
        return accel  # (N, 2)
    
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Test de l'architecture EGNN
model_egnn = EGNN(
    node_in=cfg.NODE_IN,
    node_out=cfg.NODE_OUT,
    hidden_dim=cfg.HIDDEN_DIM // 2,
    num_layers=6,
).to(DEVICE)

print("=" * 50)
print("ARCHITECTURE EGNN (E(3)-GNN)")
print("=" * 50)
print(f"Paramètres entraînables : {model_egnn.count_parameters():,}")

with torch.no_grad():
    out_egnn = model_egnn(sample_batch)
print(f"Test forward pass : x={sample_batch.x.shape} → y={out_egnn.shape} ✅")

# Test d'équivariance (rotation de π/4)
print("\n🔬 Test d'équivariance sous rotation...")
theta = np.pi / 4
R = torch.tensor([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]], dtype=torch.float, device=DEVICE)

# Données originales
out_orig = model_egnn(sample_batch).detach()

# Données tournées
sample_rot = sample_batch.clone()
sample_rot.x = sample_batch.x.clone()
sample_rot.x[:, :2] = sample_batch.x[:, :2] @ R.T  # Rotation vx, vy
out_rot = model_egnn(sample_rot).detach()

# Comparer out_rot avec (out_orig rotated)
out_orig_rot = out_orig @ R.T
equivariance_error = (out_rot - out_orig_rot).abs().mean().item()
print(f"Erreur d'équivariance : {equivariance_error:.2e} (devrait être ~0)")

---
## Section 6 — ⚛️ Architecture GMN (Physics-Informed)

In [ ]:
# ============================================================
# GMN : Graph Mechanics Network avec contraintes physiques
# ============================================================

class DivergenceFreeProjection(nn.Module):
    """
    Projection de Leray-Helmholtz pour assurer div(u) ≈ 0.
    
    Utilise une décomposition spectrale (FFT) pour projeter
    le champ de vitesse sur l'espace solénoïdal (incompressible).
    
    Pour un champ u, la projection est :
        u_div_free = u - ∇(Δ⁻¹ div(u))
    ce qui en Fourier donne :
        û_div_free = û - k(k·û)/|k|²
    """
    
    def __init__(self, Nx: int, Ny: int):
        super().__init__()
        self.Nx = Nx
        self.Ny = Ny
        
        # Nombres d'onde
        kx = torch.fft.fftfreq(Nx) * Nx  # (Nx,)
        ky = torch.fft.fftfreq(Ny) * Ny  # (Ny,)
        KX, KY = torch.meshgrid(kx, ky, indexing='ij')  # (Nx, Ny)
        K2 = KX**2 + KY**2  # |k|²
        K2[0, 0] = 1.0  # Éviter division par zéro (mode k=0)
        
        self.register_buffer('KX', KX)
        self.register_buffer('KY', KY)
        self.register_buffer('K2', K2)
    
    def forward(self, vx: torch.Tensor, vy: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            vx, vy : (B, Nx, Ny) champs de vitesse
        Returns:
            vx_pf, vy_pf : champs projetés (div-free)
        """
        # FFT 2D
        vx_hat = torch.fft.fft2(vx)
        vy_hat = torch.fft.fft2(vy)
        
        # Divergence en espace de Fourier : k · û
        div_hat = self.KX * vx_hat + self.KY * vy_hat
        
        # Correction : û_div_free = û - k(k·û)/|k|²
        correction = div_hat / self.K2
        vx_hat_pf = vx_hat - self.KX * correction
        vy_hat_pf = vy_hat - self.KY * correction
        
        # Mode k=0 : conservation de la moyenne
        vx_hat_pf[:, 0, 0] = vx_hat[:, 0, 0]
        vy_hat_pf[:, 0, 0] = vy_hat[:, 0, 0]
        
        # IFFT
        vx_pf = torch.fft.ifft2(vx_hat_pf).real
        vy_pf = torch.fft.ifft2(vy_hat_pf).real
        
        return vx_pf, vy_pf
    
    def compute_divergence(self, vx: torch.Tensor, vy: torch.Tensor) -> torch.Tensor:
        """Calcule la divergence numérique (doit être ~0)."""
        dvx_dx = torch.roll(vx, -1, dims=-2) - torch.roll(vx, 1, dims=-2)
        dvy_dy = torch.roll(vy, -1, dims=-1) - torch.roll(vy, 1, dims=-1)
        return (dvx_dx + dvy_dy).abs().mean()


class GMN(nn.Module):
    """
    Graph Mechanics Network : GNN avec contraintes d'incompressibilité.
    
    Architecture :
    1. Backbone GNN (MeshGraphNet)
    2. Prédiction des incréments de vitesse
    3. Projection div-free (Helmholtz-Hodge)
    4. Module temporel optionnel (GRU)
    """
    
    def __init__(
        self,
        node_in: int,
        node_out: int,
        hidden_dim: int = 128,
        num_gnn_layers: int = 8,
        Nx: int = 128,
        Ny: int = 128,
        use_temporal: bool = True,
    ):
        super().__init__()
        self.Nx = Nx
        self.Ny = Ny
        self.use_temporal = use_temporal
        
        # Backbone GNN
        self.gnn = MeshGraphNet(
            node_in=node_in,
            edge_in=3,
            node_out=hidden_dim,
            hidden_dim=hidden_dim,
            num_layers=num_gnn_layers,
        )
        
        # Module temporel (GRU sur les features des nœuds)
        if use_temporal:
            self.temporal = nn.GRUCell(
                input_size=hidden_dim,
                hidden_size=hidden_dim
            )
            self.h_state = None  # État caché du GRU
        
        # Décodeur final
        self.decoder = build_mlp(hidden_dim, node_out, hidden_dim, layer_norm=False)
        
        # Projection div-free
        self.div_free_proj = DivergenceFreeProjection(Nx, Ny)
        
        # Coefficient appris pour le poids de la projection
        self.proj_weight = nn.Parameter(torch.tensor(0.5))
    
    def reset_state(self):
        """Réinitialise l'état du GRU (début de nouvelle trajectoire)."""
        self.h_state = None
    
    def forward(self, data: Data) -> torch.Tensor:
        # 1. Extraction de features via GNN
        # Modifier le décodeur temporairement pour sortir hidden
        features = self.gnn(data)  # (N, hidden)
        
        # 2. Module temporel (GRU)
        if self.use_temporal:
            if self.h_state is None or self.h_state.shape[0] != features.shape[0]:
                self.h_state = torch.zeros_like(features)
            features = self.temporal(features, self.h_state)
            self.h_state = features.detach()  # Détach pour éviter TBPTT complet
        
        # 3. Décodage des incréments de vitesse
        delta_v = self.decoder(features)  # (N, 2) : (dvx, dvy)
        
        # 4. Projection div-free
        dvx = delta_v[:, 0].view(-1, self.Nx, self.Ny)
        dvy = delta_v[:, 1].view(-1, self.Nx, self.Ny)
        
        # Mélange interpolé entre avec et sans projection
        w = torch.sigmoid(self.proj_weight)
        dvx_pf, dvy_pf = self.div_free_proj(dvx.unsqueeze(0), dvy.unsqueeze(0))
        dvx_final = w * dvx_pf.squeeze(0) + (1-w) * dvx
        dvy_final = w * dvy_pf.squeeze(0) + (1-w) * dvy
        
        # Reshape en vecteur de nœuds
        accel = torch.stack([
            dvx_final.flatten(),
            dvy_final.flatten()
        ], dim=-1)
        
        return accel
    
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Test de l'architecture GMN
model_gmn = GMN(
    node_in=cfg.NODE_IN,
    node_out=cfg.NODE_OUT,
    hidden_dim=cfg.HIDDEN_DIM,
    num_gnn_layers=8,
    Nx=cfg.NX,
    Ny=cfg.NY,
).to(DEVICE)

print("=" * 50)
print("ARCHITECTURE GMN")
print("=" * 50)
print(f"Paramètres entraînables : {model_gmn.count_parameters():,}")

with torch.no_grad():
    out_gmn = model_gmn(sample_batch)
print(f"Test forward pass : x={sample_batch.x.shape} → y={out_gmn.shape} ✅")

# Vérification de la divergence
print(f"Poids de projection appris : {torch.sigmoid(model_gmn.proj_weight).item():.3f}")

---
## Section 7 — 🏋️ Entraînement et Validation

In [ ]:
# ============================================================
# Fonctions de perte et métriques
# ============================================================

def mse_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """MSE standard."""
    return F.mse_loss(pred, target)

def huber_loss(pred: torch.Tensor, target: torch.Tensor, delta: float = 1.0) -> torch.Tensor:
    """Perte de Huber (robuste aux outliers)."""
    return F.huber_loss(pred, target, delta=delta)

def compute_nmse(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-8) -> float:
    """Normalized MSE."""
    return (F.mse_loss(pred, target) / (target.var() + eps)).item()


# ============================================================
# Boucle d'entraînement générique
# ============================================================

def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    grad_clip: float = 1.0,
) -> Dict[str, float]:
    """Entraîne le modèle sur une époque."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        pred = model(batch)   # (N, 2)
        target = batch.y      # (N, 2)
        
        loss = mse_loss(pred, target)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return {'loss': total_loss / n_batches}


@torch.no_grad()
def validate_epoch(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Dict[str, float]:
    """Évalue le modèle sur le dataset de validation."""
    model.eval()
    total_loss = 0.0
    total_nmse = 0.0
    n_batches = 0
    
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch)
        target = batch.y
        
        total_loss += mse_loss(pred, target).item()
        total_nmse += compute_nmse(pred, target)
        n_batches += 1
    
    return {
        'val_loss': total_loss / n_batches,
        'val_nmse': total_nmse / n_batches,
    }


# ============================================================
# Entraînement complet
# ============================================================

def train_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Config,
    device: torch.device,
) -> Dict[str, List[float]]:
    """Entraîne un modèle GNN et retourne l'historique."""
    
    optimizer = Adam(model.parameters(), lr=config.LR, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.EPOCHS, eta_min=1e-6)
    
    history = {'train_loss': [], 'val_loss': [], 'val_nmse': [], 'lr': []}
    best_val_loss = float('inf')
    save_path = f"{config.SAVE_DIR}/{model_name}_best.pt"
    
    print(f"\n{'='*60}")
    print(f"Entraînement de {model_name}")
    print(f"{'='*60}")
    print(f"Paramètres : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Epochs : {config.EPOCHS} | LR : {config.LR} | Batch : {config.BATCH_SIZE}")
    
    start_time = time.time()
    
    pbar = tqdm(range(1, config.EPOCHS + 1), desc=model_name)
    for epoch in pbar:
        # Entraînement
        train_metrics = train_epoch(model, train_loader, optimizer, device, config.GRAD_CLIP)
        scheduler.step()
        
        # Validation périodique
        if epoch % config.LOG_EVERY == 0 or epoch == config.EPOCHS:
            val_metrics = validate_epoch(model, val_loader, device)
            
            history['train_loss'].append(train_metrics['loss'])
            history['val_loss'].append(val_metrics['val_loss'])
            history['val_nmse'].append(val_metrics['val_nmse'])
            history['lr'].append(scheduler.get_last_lr()[0])
            
            pbar.set_postfix({
                'train': f"{train_metrics['loss']:.4f}",
                'val': f"{val_metrics['val_loss']:.4f}",
                'NMSE': f"{val_metrics['val_nmse']:.4f}",
            })
            
            # Sauvegarde du meilleur modèle
            if val_metrics['val_loss'] < best_val_loss:
                best_val_loss = val_metrics['val_loss']
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_loss': best_val_loss,
                }, save_path)
    
    elapsed = time.time() - start_time
    print(f"\n✅ Entraînement terminé en {elapsed:.0f}s")
    print(f"   Meilleure val_loss : {best_val_loss:.6f}")
    print(f"   Checkpoint sauvegardé : {save_path}")
    
    return history

print("✅ Fonctions d'entraînement définies")

In [ ]:
# ============================================================
# Lancement de l'entraînement
# (Réduire EPOCHS pour les tests rapides)
# ============================================================

# Configuration rapide pour démonstration
cfg_fast = Config()
cfg_fast.EPOCHS = 20       # Réduire pour test rapide (utiliser 100+ pour de vrais résultats)
cfg_fast.LOG_EVERY = 5
cfg_fast.HIDDEN_DIM = 64   # Réduire pour test rapide
cfg_fast.NUM_LAYERS = 6

# Recréer les modèles avec la config rapide
models = {
    'MeshGraphNet': MeshGraphNet(
        node_in=cfg_fast.NODE_IN, edge_in=cfg_fast.EDGE_IN,
        node_out=cfg_fast.NODE_OUT,
        hidden_dim=cfg_fast.HIDDEN_DIM, num_layers=cfg_fast.NUM_LAYERS
    ).to(DEVICE),
    'EGNN': EGNN(
        node_in=cfg_fast.NODE_IN, node_out=cfg_fast.NODE_OUT,
        hidden_dim=cfg_fast.HIDDEN_DIM//2, num_layers=4
    ).to(DEVICE),
    'GMN': GMN(
        node_in=cfg_fast.NODE_IN, node_out=cfg_fast.NODE_OUT,
        hidden_dim=cfg_fast.HIDDEN_DIM, num_gnn_layers=6,
        Nx=cfg_fast.NX, Ny=cfg_fast.NY, use_temporal=False
    ).to(DEVICE),
}

# Entraîner tous les modèles
all_histories = {}
for name, model in models.items():
    history = train_model(
        model=model,
        model_name=name,
        train_loader=train_loader,
        val_loader=val_loader,
        config=cfg_fast,
        device=DEVICE,
    )
    all_histories[name] = history

In [ ]:
# ============================================================
# Visualisation des courbes d'apprentissage
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'MeshGraphNet': '#2196F3', 'EGNN': '#4CAF50', 'GMN': '#FF5722'}

for name, history in all_histories.items():
    epochs = [i * cfg_fast.LOG_EVERY for i in range(1, len(history['train_loss'])+1)]
    c = colors[name]
    
    axes[0].semilogy(epochs, history['train_loss'], color=c, label=name, marker='o', ms=4)
    axes[1].semilogy(epochs, history['val_loss'],  color=c, label=name, marker='s', ms=4)
    axes[2].plot(epochs, history['val_nmse'], color=c, label=name, marker='^', ms=4)

axes[0].set_title('Perte d\'entraînement'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Perte de validation');   axes[1].set_xlabel('Epoch')
axes[2].set_title('NMSE de validation');    axes[2].set_xlabel('Epoch')

for ax in axes:
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Courbes d\'apprentissage — Comparaison des architectures', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 8 — 📊 Évaluation et Visualisation

In [ ]:
# ============================================================
# Rollout autoregressif
# ============================================================

@torch.no_grad()
def rollout(
    model: nn.Module,
    initial_vx: np.ndarray,
    initial_vy: np.ndarray,
    initial_p: np.ndarray,
    edge_index: torch.Tensor,
    edge_attr: torch.Tensor,
    T_rollout: int,
    dt: float,
    norm_vx: Normalizer,
    norm_vy: Normalizer,
    norm_p: Normalizer,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Simulation autorégressive sur T_rollout pas de temps.
    
    Returns:
        pred_vx : (T_rollout, Nx, Ny) prédictions de vx
        pred_vy : (T_rollout, Nx, Ny) prédictions de vy
    """
    model.eval()
    Nx, Ny = initial_vx.shape
    
    # Initialisation
    curr_vx = norm_vx.transform(initial_vx).flatten()
    curr_vy = norm_vy.transform(initial_vy).flatten()
    curr_p  = norm_p.transform(initial_p).flatten()
    
    pred_vx_list = [initial_vx.copy()]
    pred_vy_list = [initial_vy.copy()]
    
    # Reset état GRU si GMN
    if hasattr(model, 'reset_state'):
        model.reset_state()
    
    for t in range(T_rollout):
        # Construction du graphe
        x = np.stack([curr_vx, curr_vy, curr_p], axis=1).astype(np.float32)
        data = Data(
            x=torch.from_numpy(x),
            edge_index=edge_index,
            edge_attr=edge_attr,
        ).to(device)
        
        # Prédiction de l'accélération
        accel = model(data).cpu().numpy()  # (N, 2)
        
        # Intégration Euler
        new_vx = curr_vx + accel[:, 0] * dt
        new_vy = curr_vy + accel[:, 1] * dt
        
        # Mise à jour de l'état
        curr_vx = new_vx
        curr_vy = new_vy
        curr_p  = -0.5 * (new_vx**2 + new_vy**2)  # Approximation Bernoulli
        
        # Dénormalisation et sauvegarde
        vx_phys = norm_vx.inverse_transform(new_vx.reshape(Nx, Ny))
        vy_phys = norm_vy.inverse_transform(new_vy.reshape(Nx, Ny))
        pred_vx_list.append(vx_phys)
        pred_vy_list.append(vy_phys)
    
    return np.stack(pred_vx_list), np.stack(pred_vy_list)


print("✅ Fonction de rollout définie")

In [ ]:
# ============================================================
# Métriques d'évaluation
# ============================================================

def compute_rollout_metrics(
    pred_vx: np.ndarray,
    pred_vy: np.ndarray,
    true_vx: np.ndarray,
    true_vy: np.ndarray,
    threshold: float = 0.15,
) -> Dict:
    """Calcule toutes les métriques d'évaluation sur un rollout."""
    T = min(len(pred_vx), len(true_vx))
    
    nmse_curve = []
    corr_curve = []
    ke_pred, ke_true = [], []
    
    for t in range(T):
        # NMSE
        nmse = (((pred_vx[t] - true_vx[t])**2 + (pred_vy[t] - true_vy[t])**2).mean() /
                ((true_vx[t]**2 + true_vy[t]**2).mean() + 1e-8))
        nmse_curve.append(nmse)
        
        # Corrélation
        corr_vx = np.corrcoef(pred_vx[t].flatten(), true_vx[t].flatten())[0,1]
        corr_vy = np.corrcoef(pred_vy[t].flatten(), true_vy[t].flatten())[0,1]
        corr_curve.append(0.5 * (corr_vx + corr_vy))
        
        # Énergie cinétique
        ke_pred.append(0.5 * (pred_vx[t]**2 + pred_vy[t]**2).mean())
        ke_true.append(0.5 * (true_vx[t]**2 + true_vy[t]**2).mean())
    
    nmse_curve = np.array(nmse_curve)
    corr_curve = np.array(corr_curve)
    
    # Temps de validité
    exceeded = np.where(nmse_curve > threshold)[0]
    valid_time = exceeded[0] if len(exceeded) > 0 else T
    
    return {
        'nmse_curve': nmse_curve,
        'corr_curve': corr_curve,
        'ke_pred': np.array(ke_pred),
        'ke_true': np.array(ke_true),
        'valid_time': valid_time,
        'final_nmse': nmse_curve[-1],
        'mean_nmse': nmse_curve.mean(),
    }


# Évaluation de tous les modèles
T_ROLLOUT = min(30, val_data['velocity_x'].shape[1] - 1)
traj_eval = 0  # Trajectoire de test

results = {}
for name, model in models.items():
    print(f"Rollout {name}...", end=' ')
    t0 = time.time()
    
    pred_vx, pred_vy = rollout(
        model=model,
        initial_vx=val_data['velocity_x'][traj_eval, 0],
        initial_vy=val_data['velocity_y'][traj_eval, 0],
        initial_p=val_data['pressure'][traj_eval, 0],
        edge_index=EDGE_INDEX,
        edge_attr=EDGE_ATTR,
        T_rollout=T_ROLLOUT,
        dt=cfg.DT,
        norm_vx=norm_vx,
        norm_vy=norm_vy,
        norm_p=norm_p,
        device=DEVICE,
    )
    
    metrics = compute_rollout_metrics(
        pred_vx, pred_vy,
        val_data['velocity_x'][traj_eval, :T_ROLLOUT+1],
        val_data['velocity_y'][traj_eval, :T_ROLLOUT+1],
    )
    metrics['pred_vx'] = pred_vx
    metrics['pred_vy'] = pred_vy
    metrics['rollout_time'] = time.time() - t0
    results[name] = metrics
    
    print(f"NMSE moyen={metrics['mean_nmse']:.4f}, t*={metrics['valid_time']}, ({metrics['rollout_time']:.1f}s)")

In [ ]:
# ============================================================
# Visualisation des résultats
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Évaluation des Modèles GNN — Rollout Autoregressif', fontsize=13, fontweight='bold')

colors = {'MeshGraphNet': '#2196F3', 'EGNN': '#4CAF50', 'GMN': '#FF5722'}
t_axis = np.arange(T_ROLLOUT + 1)

# 1. Courbe NMSE
ax = axes[0, 0]
for name, res in results.items():
    ax.semilogy(t_axis, res['nmse_curve'], color=colors[name], label=name, lw=2)
ax.axhline(0.15, color='red', ls='--', label='Seuil t*')
ax.set_xlabel('Pas de temps'); ax.set_ylabel('NMSE')
ax.set_title('NMSE au cours du rollout')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 2. Corrélation temporelle
ax = axes[0, 1]
for name, res in results.items():
    ax.plot(t_axis, res['corr_curve'], color=colors[name], label=name, lw=2)
ax.axhline(0.85, color='red', ls='--', label='Seuil corr.')
ax.set_xlabel('Pas de temps'); ax.set_ylabel('Corrélation')
ax.set_title('Corrélation temporelle')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 3. Énergie cinétique
ax = axes[0, 2]
ax.plot(t_axis, results['MeshGraphNet']['ke_true'], 'k-', label='CFD (vérité)', lw=2)
for name, res in results.items():
    ax.plot(t_axis, res['ke_pred'], color=colors[name], label=name, ls='--', lw=1.5)
ax.set_xlabel('Pas de temps'); ax.set_ylabel('Énergie cinétique')
ax.set_title('Conservation de l\'énergie cinétique')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 4-6. Champs de vitesse à t=T/2
t_show = T_ROLLOUT // 2
true_vx = val_data['velocity_x'][traj_eval, t_show]

vmin = true_vx.min(); vmax = true_vx.max()

for col, (name, res) in enumerate(results.items()):
    ax = axes[1, col]
    pred = res['pred_vx'][t_show]
    
    # Affichage côte à côte : vérité (haut) vs prédiction (bas)
    combined = np.concatenate([true_vx, pred], axis=1)
    im = ax.imshow(combined.T, cmap='RdBu_r', origin='lower',
                   vmin=vmin, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.7)
    ax.set_title(f'{name}\nGauche: CFD | Droite: GNN (t={t_show})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    
    # Barre de séparation
    mid = combined.shape[0] // 2
    ax.axvline(x=mid, color='yellow', lw=2, ls='--')

plt.tight_layout()
plt.savefig('./results/figures/evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Tableau de synthèse des résultats
# ============================================================

import pandas as pd

summary = []
for name, res in results.items():
    n_params = models[name].count_parameters() if hasattr(models[name], 'count_parameters') else \
               sum(p.numel() for p in models[name].parameters())
    summary.append({
        'Modèle': name,
        'Paramètres': f"{n_params:,}",
        'NMSE moyen': f"{res['mean_nmse']:.4f}",
        'NMSE final': f"{res['final_nmse']:.4f}",
        't* (pas de temps)': res['valid_time'],
        'Corr. finale': f"{res['corr_curve'][-1]:.4f}",
        'Temps rollout (s)': f"{res['rollout_time']:.2f}",
    })

df = pd.DataFrame(summary)
print("\n" + "=" * 70)
print("TABLEAU DE SYNTHÈSE DES RÉSULTATS")
print("=" * 70)
print(df.to_string(index=False))
df.to_csv('./results/summary_results.csv', index=False)
print(f"\n✅ Résultats sauvegardés dans ./results/summary_results.csv")

In [ ]:
# ============================================================
# Animation du rollout (optionnel, peut être long)
# ============================================================

def create_rollout_animation(
    results: Dict,
    true_vx: np.ndarray,
    save_path: str = './results/figures/rollout_animation.gif',
    interval: int = 100,
):
    """Crée une animation comparative du rollout."""
    model_names = list(results.keys())
    T = len(true_vx)
    
    fig, axes = plt.subplots(1, len(model_names) + 1, figsize=(5*(len(model_names)+1), 4))
    
    vmin, vmax = true_vx.min(), true_vx.max()
    
    ims_true = [axes[0].imshow(true_vx[0].T, cmap='RdBu_r', origin='lower',
                               vmin=vmin, vmax=vmax, animated=True)]
    axes[0].set_title('CFD (vérité ground truth)')
    
    ims_pred = []
    for k, name in enumerate(model_names):
        im = axes[k+1].imshow(
            results[name]['pred_vx'][0].T, cmap='RdBu_r', origin='lower',
            vmin=vmin, vmax=vmax, animated=True
        )
        axes[k+1].set_title(name)
        ims_pred.append(im)
    
    title = fig.suptitle('t = 0', fontsize=12)
    
    def update(t):
        ims_true[0].set_array(true_vx[min(t, len(true_vx)-1)].T)
        for k, name in enumerate(model_names):
            pred = results[name]['pred_vx']
            ims_pred[k].set_array(pred[min(t, len(pred)-1)].T)
        title.set_text(f't = {t}')
        return [ims_true[0]] + ims_pred + [title]
    
    T_anim = min(T, len(results[model_names[0]]['pred_vx']))
    anim = animation.FuncAnimation(
        fig, update, frames=T_anim,
        interval=interval, blit=False
    )
    
    anim.save(save_path, writer='pillow', fps=10, dpi=80)
    plt.close()
    print(f"✅ Animation sauvegardée : {save_path}")


# Créer l'animation (commenter si trop lent)
create_rollout_animation(
    results,
    true_vx=val_data['velocity_x'][traj_eval, :T_ROLLOUT+1],
    save_path='./results/figures/rollout_animation.gif'
)

In [ ]:
# ============================================================
# Section Bonus : Étude d'Ablation et Généralisation
# ============================================================

print("="*60)
print("PISTES D'EXPÉRIENCES SUPPLÉMENTAIRES")
print("="*60)

experiments = [
    ("Ablation : nombre de couches", 
     "Tester L ∈ {4, 6, 8, 10, 15} sur MeshGraphNet"),
    ("Ablation : dimension cachée",
     "Tester d ∈ {64, 128, 256} et tracer Pareto perf/paramètres"),
    ("Généralisation Re",
     "Entraîner sur Re < 1000, tester sur Re > 2000"),
    ("Généralisation résolution",
     "Entraîner sur 64x64, évaluer sur 128x128 (zero-shot)"),
    ("Projection div-free",
     "Comparer GMN avec/sans projection Helmholtz-Hodge"),
    ("Schéma d'intégration",
     "Comparer Euler vs RK4 pour le rollout"),
    ("Bruit d'entraînement",
     "Étude de l'effet de σ_noise sur la robustesse du rollout"),
]

for i, (title, desc) in enumerate(experiments, 1):
    print(f"\n  {i:02d}. {title}")
    print(f"      → {desc}")

print("\n" + "="*60)
print("✅ NOTEBOOK COMPLET — Bonne implémentation !")
print("="*60)